In [ ]:
from typing import Any, Callable, Optional, Union

from pprint import pprint
from datetime import datetime
from pathlib import Path
import os
import re

from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import pytz
import numpy as np
import safetensors.torch as safetensors
import tqdm.notebook as tqdm
import torch
import torch.utils.data as torchdata
import torch.nn as nn
import torch.nn.functional as torchfunc
import torchmetrics
import torchvision
import torchvision.transforms as torchtrans

# import segmentation_models_pytorch as torchseg
import yaml

from flatiron.core.dataset import Dataset
from flatiron.core.types import Compiled, Filepath, Getter
from flatiron.core.tools import get_tensorboard_project
from flatiron.torch.tools import ModelCheckpoint, get_callbacks, TorchDataset, _execute_epoch

Filepath = Union[str, Path]

# QUADRO P6000 is CUDA 6.1
# Triton is CUDA 7.0+
# This tells triton to shutup
import torch._dynamo
torch._dynamo.config.suppress_errors = True

In [ ]:
def train(
    device,      # type: str
    model,       # type: torch.nn.Module
    optimizer,   # type: torch.optim.Optimizer
    loss,        # type: torch.nn.Module
    metrics,     # type: list[torch.nn.Module]
    callbacks,   # type: Callbacks
    train_data,  # type: Dataset
    test_data,   # type: Dataset
    params,      # type: dict
):
    # type: (...) -> None
    '''
    Train Torch model.

    Args:
        device (str): Device to compile to.
        model (torch.nn.Module): Model to be compiled.
        optimizer (dict): Optimizer config for compilation.
        loss (str): Loss to be compiled.
        metrics (list[str]): Metrics function to be compiled.
        callbacks (dict): Dict of callbacks.
        train_data (Dataset): Training dataset.
        test_data (Dataset): Test dataset.
        params (dict): Training params.
    '''
    checkpoint = callbacks['checkpoint']  # type: Any
    writer = callbacks['tensorboard']
    batch_size = params['batch_size']

    device = torch.device(device)
    torch.manual_seed(params['seed'])
    model = model.to(device)
    loss = loss.to(device)
    metrics = [x.to(device) for x in metrics]
    
    train_loader = torch.utils.data.DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
    )
    test_loader = torch.utils.data.DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
    )

    kwargs = dict(
        model=model,
        optimizer=optimizer,
        loss_func=loss,
        device=device,
        metrics_funcs=metrics,
        writer=writer,
    )
    for i in tqdm.trange(params['epochs']):
        _execute_epoch(
            epoch=i, mode='train', data_loader=train_loader,
            checkpoint=checkpoint, **kwargs
        )
        _execute_epoch(epoch=i, mode='test', data_loader=test_loader, **kwargs)
        if checkpoint.save_freq == 'epoch':
            checkpoint.save(model, i)

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, fc_size=40, num_classes=10, hidden_blocks=2):
        super().__init__()
        kwargs = dict(kernel_size=(3, 3))
        
        self.layers = nn.ModuleList()
        self.layers.extend([
            nn.ModuleDict(dict(
                input=nn.Conv2d(3, 10, **kwargs),
            ))
        ])
        for _ in range(hidden_blocks):
            self.layers.extend([
                nn.ModuleDict(dict(
                    conv=nn.Conv2d(10, 10, **kwargs),
                    act=nn.ReLU(),
                    pool=nn.MaxPool2d(**kwargs),
                ))
            ])
        self.layers.extend([
            nn.ModuleDict(dict(
                flatten=nn.Flatten(),
                full=nn.Linear(fc_size, out_features=num_classes),
                output=nn.Softmax(dim=-1),
            )),
        ])
    
    def forward(self, x, index=None):
        for i, layer in enumerate(self.layers):
            for name, module in layer.items():
                x = module(x)
                if index == f'{i}.{name}':
                    return x
        return x

In [ ]:
transform = torchtrans.Compose([
    torchtrans.ToTensor(),
    torchtrans.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    ),
])

root = '/mnt/storage/data/cifar10'
os.makedirs(root, exist_ok=True)
batch_size = 4
train_data = torchvision.datasets.CIFAR10(
    root=root,
    train=True,
    download=True,
    transform=transform,
)

test_data = torchvision.datasets.CIFAR10(
    root=root,
    train=False,
    download=True,
    transform=transform,
)


classes = (
    'plane', 'car', 'bird', 'cat', 'deer', 'dog',
    'frog', 'horse', 'ship', 'truck'
)

In [ ]:
droot = '/mnt/storage/projects'
project = 'cifar001'
p = Path(droot, project)
os.makedirs(p, exist_ok=True)

In [ ]:
model = SimpleClassifier()

# CALLBACKS
tb = get_tensorboard_project(
    project=project,
    root='/mnt/storage/projects',
    extension='pth',
    timezone='America/Detroit',
)
print('TENSORBOARD')
pprint(tb)

callback_kwargs = dict(
    log_directory=tb['log_dir'],
    checkpoint_pattern=tb['checkpoint_pattern'],
    checkpoint_params=dict(save_freq='epoch'),
)
callbacks = get_callbacks(**callback_kwargs)
print()
print('CALLBACKS')
pprint(callback_kwargs)

metric_kwargs = dict(
    task='multiclass',
    num_classes=len(classes),
)

# TRAIN KWARGS
train_kwargs = dict(
    device='cuda',
    model=torch.compile(model),
    optimizer=torch.optim.SGD(
        model.parameters(),
        # filter(lambda x: x.requires_grad, model.parameters()),
        lr=0.002,
        momentum=0.9,
    ),
    loss=nn.CrossEntropyLoss(),
    metrics=[
        torchmetrics.classification.F1Score(**metric_kwargs),
        torchmetrics.classification.Precision(**metric_kwargs),
        torchmetrics.classification.Recall(**metric_kwargs),
        torchmetrics.classification.Specificity(**metric_kwargs),
    ],
    callbacks=callbacks,
    train_data=train_data,
    test_data=test_data,
    params=dict(
        epochs=10,
        seed=42,
        batch_size=batch_size,
    )
)
print()
print('TRAIN')
pprint(train_kwargs)

# TRAIN
print()
train(**train_kwargs)